In [2]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [3]:
# # downloaded project inputs
# input_data_path, build_files_path = preprocess.unpack_files(data_files = '/data2/hratch/human_me/data.zip', 
#                                         build_files = '/data2/hratch/human_me/build_files.zip', 
#                                         data_out = '/data2/hratch/human_me/raw')
# preprocess.create_environment(input_data_path, build_files_path, root_path = '/home/hratch/Projects/human_me/',
#                               processed_data_path = '/data2/hratch/human_me/processed/', 
#                               n_cores = 20)

In [4]:
from preprocess import correct_inputs 

full model

In [5]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/recon2_2.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')
# # # optional - only if you want to express non-machinery proteins
# # correct_inputs.check_non_machinery(nonmachinery_file = '/data2/hratch/human_me/input_files/non_machinery.txt')

# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = False, compress_mrna = False)

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'me_model.pickle', 'wb') as handle:
#     pickle.dump(me_model, handle)

toy model

In [6]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/toy_model.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
#                                                 model_id = 'toy_me_model')

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'toy_me_model.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)

# Assess changes

In [7]:
import h5py
import numpy as np
import pandas as pd
import pickle

In [8]:
from expression import build_me_model
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                model_id = 'toy_me_model')

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degrdation
Generate ribosome


../scripts/expression/protein_expression/ubiquitin.py:34 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/protein_expression/ubiquitin.py:64 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/gene_information.py:114 UserWarning: HGNC:10368: The letter X is in the protein sequence. Replacing with a random amino acid
  0%|          | 0/591 [00:00<?, ?it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:13<00:00, 43.67it/s]


...

A total of 1897 reactions were dropped when forming a minimal proteome
Add machinery to metabolic module reactions


  0%|          | 33/10659 [00:00<00:33, 317.81it/s]

Add machinery to expression module reactions


100%|██████████| 10659/10659 [00:45<00:00, 235.14it/s]


Generate ME-Model
Time to build: 4.46021765867869 minutes


In [ ]:
# RPL40 with nuclear degradation: infeasible
# RPL40 AND all RBs nuclear degradation: infeasible
# RPL40 no nuclear degradation, RBs nuclear degradation: infeasible
# RPL40 part of regular RBs, all nuclear degraded: infeasible
# RPL40 part of regular RBs, no nuclear degradataion: 
# more products generated from cleavage reaction (stoichiometric change)

In [10]:
counter = 3
base = 2 # automate trna biomass (leads to 10e-15 minute changes in biomass value)
# base = 4 # removed Protein.mass, automated mrna biomass change
# base = 3 # changed to compress_mrna = False

In [11]:
sln, stat,_ = toy_me_model.solve_lp(1e-9)

fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
if stat == 0:
    S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    raise ValueError('Model did not solve')

/home/hratch/anaconda3/envs/CD8T_RA/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '2'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


Last saved file: 2


In [13]:
S_1 = pd.read_hdf(fn, key = str(counter))
S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
            raise ValueError('Dataframes are not equal due to stoichiometric values mismatch')
        else:
            print('Success, please update git')
            lp_path = '/data2/hratch/human_me/test_lp/'
            with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'wb') as handle:
                pickle.dump(toy_me_model, handle)
    else:
        raise ValueError('Dataframes are not equal due to column/row label mismatch')
else:
    print('Success')
    lp_path = '/data2/hratch/human_me/test_lp/'
    with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'wb') as handle:
        pickle.dump(toy_me_model, handle)

In [14]:
am = S_1.index.tolist()
mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
for m in mm:
    mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

am, rm = S_1.index.tolist(), S_1.columns.tolist()
mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
for m in mm_2:
    mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))




In [15]:
mm.keys()

dict_keys([774, 1992, 1994, 1996, 2667, 2668, 10048])

In [40]:
m_idx = 10048
mm_2[m_idx]

{'id': 'biomass_unmodeled_protein',
 'reactions': ['folded_protein_c_DEUBIQUITINATIONc']}

In [41]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:DUMMY_folded_protein_c_DEUBIQUITINATIONc    0.018015
Name: biomass_unmodeled_protein, dtype: float64

In [42]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:DUMMY_folded_protein_c_DEUBIQUITINATIONc    0.018015
Name: biomass_unmodeled_protein, dtype: float64

In [43]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-8]

Series([], Name: biomass_unmodeled_protein, dtype: float64)

In [49]:
lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'working_version_' + str(base) + '.pickle', 'rb') as handle:
    tme2 = pickle.load(handle)

In [54]:
toy_me_model.reactions.get_by_id('HGNC:12458_UBIQUITIN_CLEAVAGEc').metabolites

{<Protein HGNC:12458_processed_unfolded_protein_c at 0x7fee1003add8>: 1,
 <Protein HGNC:12515_folded_protein_c at 0x7fee59efdbe0>: -0.00015625861864873*mu - 3.1251723729746e-6,
 <Protein HGNC:12458_unfolded_protein_c at 0x7fee1003ad68>: -1,
 <Metabolite h2o_c at 0x7fee03779cc0>: -1,
 <Protein ubiquitin_monomer_protein_c at 0x7fee0a280470>: 1,
 <Metabolite biomass_protein at 0x7fee0a280da0>: 0.018015280000001077}

In [55]:
tme2.reactions.get_by_id('HGNC:12458_UBIQUITIN_CLEAVAGEc').metabolites

{<Protein HGNC:12458_processed_unfolded_protein_c at 0x7fee3aa0dfd0>: 2,
 <Protein HGNC:12515_folded_protein_c at 0x7fee3aa0d668>: -0.00015625861864873*mu - 3.1251723729746e-6,
 <Protein HGNC:12458_unfolded_protein_c at 0x7fee3aa31860>: -2,
 <Metabolite h2o_c at 0x7fee00272828>: -2,
 <Protein ubiquitin_monomer_protein_c at 0x7fee2b515c18>: 2,
 <Metabolite biomass_protein at 0x7fee173edba8>: -17.111457840000003}

# testing ubiquitin cleavage

In [45]:
from utils import utils_2
from utils import functions as func
from tqdm import tqdm
from utils import parameters as params
import copy
from utils import metabolites as metab
import pickle
import sympy

lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
    tme = pickle.load(handle)
# 17.111457840000003

In [37]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    


In [40]:
all_fluxes = dict()
key_fluxes = dict()
for biomass_val in [-17, -8, -0.36, -0.25]:
    new_reactions = [r.copy() for r in tqdm(tme.reactions)]
    r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
    r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                     combine = False)
    test_me = func.ME_Model('test')
    test_me.add_reactions(new_reactions)
    print('Begin solve')
    sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
    
    all_fluxes[biomass_val] = dict(zip([r.id for r in tme.reactions], list(sln0[:len(tme.reactions)])))
    key_fluxes[biomass_val] = {r.id: sln0[tme.reactions.index(r.id)] for r in tme.reactions if 'HGNC:12458' in r.id or 'biomass' in r.id}

In [53]:
new_reactions = [r.copy() for r in tqdm(tme.reactions)]
r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
r_.add_metabolites({tme.metabolites.get_by_id('HGNC:12458_processed_unfolded_protein_c'): 2, 
                   tme.metabolites.get_by_id('ubiquitin_monomer_protein_c'): 2}, 
                 combine = False)
test_me = func.ME_Model('test')
test_me.add_reactions(new_reactions)
print('Begin solve')
sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)

100%|██████████| 12611/12611 [01:47<00:00, 116.79it/s]


Begin solve
Getting MINOS parameters...
Done in 177.193 seconds with status 0


In [49]:
tme.reactions.get_by_id('HGNC:12458_UBIQUITIN_CLEAVAGEc').metabolites

{<Protein HGNC:12458_processed_unfolded_protein_c at 0x7fa1896784e0>: 1,
 <Protein HGNC:12515_folded_protein_c at 0x7fa189678518>: -0.00015625861864873*mu - 3.1251723729746e-6,
 <Protein HGNC:12458_unfolded_protein_c at 0x7fa189678e48>: -1,
 <Metabolite h2o_c at 0x7fa13482efd0>: -1,
 <Protein ubiquitin_monomer_protein_c at 0x7fa1479ca6a0>: 1,
 <Metabolite biomass_protein at 0x7fa1817e5710>: 0.018015280000001077}

In [73]:
for r_id, val in key_fluxes[0.018].items():
    ub = tme.reactions.get_by_id(r_id).upper_bound
    lb = tme.reactions.get_by_id(r_id).lower_bound
    
    if isinstance(ub, sympy.Expr):
        ub = float(ub.subs(params.mu, 1e-9))
    if isinstance(lb, sympy.Expr):
        lb = float(lb.subs(params.mu, 1e-9))
    if (val > ub) or (val < lb):
        print(r_id)

HGNC:12458_processed_folded_protein_c_DEUBIQUITINATIONc


In [ ]:
#HGNC:12458_UBIQUITIN_CLEAVAGEc was incorrect with -17 (even x2), it should be correct with 0.018

In [ ]:
# row_mapper = dict(zip(sorted(set(S_1.index).difference(S_0.index)), sorted(set(S_0.index).difference(S_1.index))))
# S_1.rename(index = row_mapper,inplace = True)

# rxn_name = S_0.columns[mismatch[0][1]]
# print('rxn: ' + rxn_name)
# print('metabolite: ' + S_0.index[mismatch[0][0]])
# print('orign value: {}'.format(S_0.iloc[tuple(mismatch[0])]))
# print('new value: {}'.format(S_1.iloc[tuple(mismatch[0])]))

In [ ]:
# original
# due to a difference in complex biomass (precision issue)

# rxn: 60s_maturation
# metabolite: biomass_protein
# orign value: -1.1368683772161603e-13
# new value: 0.0